<a href="https://colab.research.google.com/github/humairaneha/Learning-Pytorch/blob/main/Transfer_Learning_using_PyTorch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [52]:
import torch
import torchvision
import pandas as pd
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn


In [53]:
torch.manual_seed(42)

In [54]:
if torch.cuda.is_available():
  device = torch.device('cuda')
else:
  device = torch.device('cpu')

In [55]:
df_train = pd.read_csv('fashion-mnist_train.csv')
df_test = pd.read_csv('fashion-mnist_test.csv')

In [56]:
X_train = df_train.iloc[:, 1:].values
y_train = df_train.iloc[:, 0].values
X_test = df_test.iloc[:, 1:].values
y_test = df_test.iloc[:, 0].values

The inference transforms are available at VGG16_Weights.IMAGENET1K_V1.transforms and perform the following preprocessing operations: **Accepts PIL.Image, batched (B, C, H, W) and single (C, H, W) image torch.Tensor objects. The images are resized to resize_size=[256] using interpolation=InterpolationMode.BILINEAR, followed by a central crop of crop_size=[224]. Finally the values are first rescaled to [0.0, 1.0] and then normalized using mean=[0.485, 0.456, 0.406] and std=[0.229, 0.224, 0.225].**

In [57]:
#transforms

from torchvision.transforms import transforms

custom_transform = transforms.Compose(
    [transforms.Resize(256),
     transforms.CenterCrop(224),
     transforms.ToTensor(), # does 2 task. convert image to tensor + scales value between 0 to 1
     transforms.Normalize(mean=[0.485, 0.456, 0.406],std=[0.229, 0.224, 0.225])
     ]
)

In [58]:
from PIL import Image
import numpy as np

In [59]:

class CustomDataset(Dataset):
    def __init__(self, features, labels,transform):
      self.features = features
      self.labels = torch.tensor(labels,dtype=torch.long)
      self.transform = transform

    def __len__(self):
      return self.features.shape[0]

    def __getitem__(self,idx):

      #resize to (28,28)

      image = self.features[idx].reshape(28,28)

      #change datatype to np.uint8

      image = image.astype(np.uint8)

      #change grayscale to color (H,W,C) -> but we want (C,H,W)

      image = np.stack([image]*3,axis=-1) # 3 channel

      #convert array to PIL image

      image = Image.fromarray(image)

      #apply transforms

      image = self.transform(image)

      return image , self.labels[idx]



In [60]:
train_dataset = CustomDataset(X_train,y_train,custom_transform)
test_dataset = CustomDataset(X_test,y_test,custom_transform)

In [61]:
train_loader = DataLoader(train_dataset,batch_size=32,shuffle=True)
test_loader = DataLoader(test_dataset,batch_size=32,shuffle=False)

In [62]:
import torchvision.models as models
vgg16= models.vgg16(pretrained=True)


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [63]:
vgg16.features


Sequential(
  (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (1): ReLU(inplace=True)
  (2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (3): ReLU(inplace=True)
  (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (5): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (6): ReLU(inplace=True)
  (7): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (8): ReLU(inplace=True)
  (9): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (10): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (11): ReLU(inplace=True)
  (12): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (13): ReLU(inplace=True)
  (14): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (15): ReLU(inplace=True)
  (16): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (17): Conv2d(256, 512, kernel_si

In [64]:
vgg16.classifier

Sequential(
  (0): Linear(in_features=25088, out_features=4096, bias=True)
  (1): ReLU(inplace=True)
  (2): Dropout(p=0.5, inplace=False)
  (3): Linear(in_features=4096, out_features=4096, bias=True)
  (4): ReLU(inplace=True)
  (5): Dropout(p=0.5, inplace=False)
  (6): Linear(in_features=4096, out_features=1000, bias=True)
)

In [65]:
for param in vgg16.features.parameters():
  param.requires_grad=False



In [66]:
vgg16.classifier = nn.Sequential(
    nn.Linear(25088 , 1024),
    nn.ReLU(),
    nn.Dropout(0.5),
    nn.Linear(1024,512),
    nn.ReLU(),
    nn.Dropout(0.5),
    nn.Linear(512,10)

)

In [67]:
vgg16.classifier

Sequential(
  (0): Linear(in_features=25088, out_features=1024, bias=True)
  (1): ReLU()
  (2): Dropout(p=0.5, inplace=False)
  (3): Linear(in_features=1024, out_features=512, bias=True)
  (4): ReLU()
  (5): Dropout(p=0.5, inplace=False)
  (6): Linear(in_features=512, out_features=10, bias=True)
)

In [68]:
vgg16 = vgg16.to(device)

In [73]:
learning_rate = 0.0001 # in general lr should be smaller for transfer learning
epochs=10

In [70]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(vgg16.classifier.parameters(),lr=learning_rate)

In [74]:
for epoch in range(epochs):

  total_epoch_loss = 0.0

  for batch_train, batch_labels in train_loader:

    batch_train = batch_train.to(device)
    batch_labels = batch_labels.to(device)

    output = vgg16(batch_train)

    loss = criterion(output,batch_labels)

    optimizer.zero_grad()

    loss.backward()

    optimizer.step()

    total_epoch_loss+=loss.item()

  avg_loss = total_epoch_loss/len(train_loader)

  print(f'epoch {epoch+1} loss {avg_loss}')


epoch 1 loss 0.312874240514636
epoch 2 loss 0.20996200054933628
epoch 3 loss 0.16317896439880133
epoch 4 loss 0.1279268021972229
epoch 5 loss 0.1025790970608592
epoch 6 loss 0.08287909380501757
epoch 7 loss 0.06897317886156962
epoch 8 loss 0.05675689545948601
epoch 9 loss 0.04897680364382104
epoch 10 loss 0.04261378094852165


In [76]:
vgg16.eval()
for epoch in range(epoch):

  total = 0
  correct = 0

  for batch_test, batch_labels in test_loader:
    # Move inputs and labels to the correct device
    batch_test = batch_test.to(device)
    batch_labels = batch_labels.to(device)

    output = vgg16(batch_test)

    predictions = torch.argmax(output,dim=1)

    total+= len(batch_labels)

    correct+= (batch_labels==predictions).sum().float().item()


accuracy = correct/total

print(f'accuracy {accuracy:.4f}')

accuracy 0.9350
